# Quantile barycenters of four histograms

This notebook generates `fig:barycenters-four-histograms`.  On the line, the quadratic Wasserstein barycenter is explicit in quantile coordinates: for four input laws $\beta_{ij}$ and bilinear weights
$$
\lambda_{00}=(1-u)(1-v),\quad \lambda_{10}=u(1-v),\quad
\lambda_{01}=(1-u)v,\quad \lambda_{11}=uv,
$$
the barycenter $\alpha_{u,v}$ satisfies
$$
F_{\alpha_{u,v}}^{-1}(r)=\sum_{i,j\in\{0,1\}} \lambda_{ij}(u,v)F_{\beta_{ij}}^{-1}(r).
$$
The figure keeps the computation lightweight: four smooth one-dimensional histograms are tabulated on a fixed grid, their quantile functions are interpolated once, and a $5\times5$ family of barycentric histograms is reconstructed from equal quantile samples.


In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    box_axes,
    figure_dir,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()
rng = np.random.default_rng(2028)

## Four corner laws

The corner laws are simple Gaussian mixtures on the same compact one-dimensional grid.  They are chosen only for visual variety: narrow and broad modes, unimodal and bimodal shapes, and different locations.  Each density is normalized by numerical quadrature.

In [ ]:
def normal_pdf(x, mean, std):
    return np.exp(-0.5 * ((x - mean) / std) ** 2) / (std * np.sqrt(2 * np.pi))


def mixture_pdf(x, weights, means, stds):
    pdf = np.zeros_like(x, dtype=float)
    for w, m, s in zip(weights, means, stds):
        pdf += w * normal_pdf(x, m, s)
    return pdf / np.trapezoid(pdf, x)


x_grid = np.linspace(-3.25, 3.25, 620)
corner_specs = [
    ([0.62, 0.38], [-1.75, -0.55], [0.32, 0.48]),
    ([0.48, 0.52], [0.45, 1.72], [0.28, 0.38]),
    ([0.40, 0.60], [-1.22, 0.98], [0.24, 0.36]),
    ([0.72, 0.28], [-0.12, 1.38], [0.62, 0.23]),
]
corner_pdfs = np.array([mixture_pdf(x_grid, *spec) for spec in corner_specs])
corner_names = [r"$\beta_{00}$", r"$\beta_{10}$", r"$\beta_{01}$", r"$\beta_{11}$"]
corner_colors = np.array([
    [0.839, 0.188, 0.153],  # red
    [0.992, 0.682, 0.380],  # orange
    [0.482, 0.196, 0.580],  # violet
    [0.129, 0.400, 0.675],  # blue
])

## Quantile representation

The cumulative functions are monotone on the grid, so inverse CDFs are obtained by one-dimensional interpolation.  Barycentric quantiles are then ordinary weighted averages.  To plot them as histograms, we evaluate each barycentric quantile at uniform mass levels and bin the resulting samples on a common axis.

In [ ]:
def inverse_cdf_from_pdf(x, pdf, levels):
    dx = np.diff(x)
    cdf = np.empty_like(x)
    cdf[0] = 0.0
    cdf[1:] = np.cumsum(0.5 * (pdf[:-1] + pdf[1:]) * dx)
    cdf = cdf / cdf[-1]
    return np.interp(levels, cdf, x)


def bilinear_weights(u, v):
    return np.array([(1 - u) * (1 - v), u * (1 - v), (1 - u) * v, u * v])


def bilinear_color(u, v):
    w = bilinear_weights(u, v)
    color = w @ corner_colors
    return tuple(np.clip(color, 0, 1))


def smooth_curve(y):
    kernel = np.array([1, 4, 7, 10, 7, 4, 1], dtype=float)
    kernel = kernel / kernel.sum()
    return np.convolve(y, kernel, mode="same")


levels = (np.arange(2200) + 0.5) / 2200
corner_quantiles = np.array([inverse_cdf_from_pdf(x_grid, pdf, levels) for pdf in corner_pdfs])
plot_grid = np.linspace(-3.25, 3.25, 520)
bin_centers = plot_grid


def barycenter_histogram(u, v):
    q = bilinear_weights(u, v) @ corner_quantiles
    dq = np.gradient(q, levels, edge_order=2)
    density_at_quantiles = 1.0 / np.maximum(dq, 1e-5)
    density = np.interp(plot_grid, q, density_at_quantiles, left=0.0, right=0.0)
    density = np.maximum(smooth_curve(density), 0.0)
    density = density / np.trapezoid(density, plot_grid)
    return q, density


uv_grid = np.linspace(0.0, 1.0, 5)
grid_hists = {(u, v): barycenter_histogram(u, v)[1] for v in uv_grid for u in uv_grid}
y_max = max(float(h.max()) for h in grid_hists.values())

## Exported panels

The PDFs are title-free.  The first panel shows the four input histograms in their corner positions; the second panel shows the $5\times5$ quantile barycenter grid.  Panel labels and the weight formula are added in LaTeX.

In [ ]:
fig_name = "barycenters-four-histograms"
out = figure_dir(fig_name)

# Corner-input panel: same axes for all four laws, arranged as the interpolation square.
fig, axs = plt.subplots(2, 2, figsize=(2.55, 2.35), sharex=True, sharey=True)
corner_positions = [(1, 0), (1, 1), (0, 0), (0, 1)]
for idx, ax_pos in enumerate(corner_positions):
    ax = axs[ax_pos]
    color = tuple(corner_colors[idx])
    ax.fill_between(x_grid, corner_pdfs[idx], color=color, alpha=0.18, linewidth=0)
    ax.plot(x_grid, corner_pdfs[idx], color=color, lw=1.35)
    ax.set_xlim(-3.1, 3.05)
    ax.set_ylim(0.0, 1.12 * corner_pdfs.max())
    ax.set_xticks([])
    ax.set_yticks([])
    box_axes(ax)
    for spine in ax.spines.values():
        spine.set_color("#d7d0bf")
        spine.set_linewidth(0.55)
fig.subplots_adjust(wspace=0.07, hspace=0.07)
save_pdf(fig, out / "corners.pdf", pad_inches=0.045)
plt.close(fig)

# Barycenter grid.  The bottom-left, bottom-right, top-left, and top-right
# panels reproduce the four corner input laws.
fig, axs = plt.subplots(5, 5, figsize=(5.55, 5.2), sharex=True, sharey=True)
for row, v in enumerate(uv_grid[::-1]):
    for col, u in enumerate(uv_grid):
        ax = axs[row, col]
        hist = grid_hists[(float(u), float(v))]
        color = bilinear_color(float(u), float(v))
        ax.fill_between(bin_centers, hist, color=color, alpha=0.20, linewidth=0)
        ax.plot(bin_centers, hist, color=color, lw=1.15)
        ax.axhline(0.0, color="#a99f8d", lw=0.45, alpha=0.75)
        ax.set_xlim(-3.1, 3.05)
        ax.set_ylim(0.0, 1.06 * y_max)
        ax.set_xticks([])
        ax.set_yticks([])
        box_axes(ax)
        for spine in ax.spines.values():
            spine.set_color("#d7d0bf")
            spine.set_linewidth(0.52)
fig.subplots_adjust(wspace=0.08, hspace=0.08)
save_pdf(fig, out / "grid.pdf", pad_inches=0.045)
plt.close(fig)

## Quick consistency checks

The corner weights recover the input laws, and every displayed histogram integrates to one up to binning error.

In [ ]:
masses = []
for key, hist in grid_hists.items():
    masses.append(np.trapezoid(hist, bin_centers))
print(f"mass range across displayed histograms: {min(masses):.6f} -- {max(masses):.6f}")